# 04 · Retos del workshop RAG

Workshop RAG · SI3003 · **Daniel Muñeton**

Mi version de `04_challenges_starter.ipynb` con los `TODO` resueltos.
Retos cubiertos aqui: **C1**, **C4** y **C6**. Los retos C2, C3 y C5 estan en el
notebook `03` (son extensiones del RAG de papers).

## C1 · HNSW contra fuerza bruta con FAISS

Linea base con FAISS: comparo busqueda exacta (`IndexFlatL2`) contra HNSW para
distintos `M` y `efSearch`, midiendo `recall@k` y latencia.

In [ ]:
%pip install -q faiss-cpu matplotlib

In [ ]:
import time
import faiss
import numpy as np

def experimento(n, d=64, nq=200, k=10, seed=42):
    rng = np.random.default_rng(seed)
    xb = rng.random((n, d), dtype="float32")
    xq = rng.random((nq, d), dtype="float32")

    flat = faiss.IndexFlatL2(d)
    flat.add(xb)
    t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000

    def recall_at_k(I, I_true):
        return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

    rows = []
    for M in (8, 16, 32):
        index = faiss.IndexHNSWFlat(d, M)
        index.hnsw.efConstruction = 100
        t0 = time.time(); index.add(xb); build_s = time.time() - t0
        for ef in (10, 16, 32, 64, 128, 256):
            index.hnsw.efSearch = ef
            t0 = time.time(); _, I = index.search(xq, k); ms = (time.time() - t0) / nq * 1000
            rows.append((M, ef, recall_at_k(I, I_true), ms, build_s))
    return t_flat, rows

t_flat, rows = experimento(20_000)
print(f"exacto: {t_flat:.3f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@10={rec:.3f}  {ms:.3f} ms/q  (build {b:.1f}s)")

### TODO 1 (resuelto) · Grafica recall vs efSearch, una curva por M

In [ ]:
import matplotlib.pyplot as plt

def graficar(rows, titulo):
    plt.figure(figsize=(7, 4))
    for M in sorted({r[0] for r in rows}):
        efs   = [r[1] for r in rows if r[0] == M]
        recs  = [r[2] for r in rows if r[0] == M]
        plt.plot(efs, recs, marker="o", label=f"M={M}")
    plt.xscale("log")
    plt.xlabel("efSearch (escala log)")
    plt.ylabel("recall@10")
    plt.title(titulo)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

graficar(rows, "HNSW · recall@10 vs efSearch (n=20k)")

### TODO 2 (resuelto) · Barrido de n y punto de cruce

Repito con n mayores y comparo la latencia del mejor HNSW (recall >= 0.95)
contra la busqueda exacta. El punto de cruce es el n donde HNSW empieza a ser
mas rapido que el exacto manteniendo alto recall.

> Nota: n = 500_000 puede tardar varios minutos en construir. Reduce el conjunto
> si tu runtime es limitado.

In [ ]:
resumen = []
for n in (20_000, 100_000, 500_000):
    t_flat_n, rows_n = experimento(n)
    # mejor HNSW con recall >= 0.95, el de menor latencia
    buenos = [r for r in rows_n if r[2] >= 0.95]
    if buenos:
        mejor = min(buenos, key=lambda r: r[3])
        resumen.append((n, t_flat_n, mejor[3], mejor[0], mejor[1], mejor[2]))
        print(f"n={n:>7}  exacto {t_flat_n:7.3f} ms/q  |  HNSW {mejor[3]:7.3f} ms/q "
              f"(M={mejor[0]}, ef={mejor[1]}, recall={mejor[2]:.3f})  "
              f"-> HNSW {'MAS RAPIDO' if mejor[3] < t_flat_n else 'mas lento'}")
    else:
        print(f"n={n}: ningun HNSW alcanzo recall>=0.95 en el barrido de ef")

print("\nResumen (n, exacto_ms, hnsw_ms): ", [(r[0], round(r[1],3), round(r[2],3)) for r in resumen])
print("Conclusiones detalladas en RESPUESTAS.md (C1).")

## C4 · Evaluacion automatica de la recuperacion

Uso `eval/preguntas.csv` (ampliado a 10 filas) para medir `hit@k`: una pregunta
cuenta como acierto si algun chunk recuperado contiene la palabra clave **y**
viene de la fuente esperada. Comparo `hit@3` y `hit@5` para `chunk_size`
400, 900 y 1500.

> Requiere las variables del notebook 03 (`textos`, `TITULOS`, `build_store`,
> `pdf_to_documents`). Ejecuta primero las partes 0-3 del notebook 03, o pega
> aqui esas definiciones.

In [ ]:
import csv
from pathlib import Path

def cargar_preguntas(ruta="eval/preguntas.csv"):
    ruta = Path(ruta)
    if not ruta.exists():
        ruta = Path("si3003-workshop-rag-Danimuneton/eval/preguntas.csv")
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def hit_at_k(retriever, filas):
    hits = 0
    for r in filas:
        docs = retriever.invoke(r["pregunta"])
        ok = any(
            r["palabra_clave"].lower() in d.page_content.lower()
            and d.metadata["source"] == r["fuente_esperada"]
            for d in docs
        )
        hits += ok
    return hits / len(filas)

filas = cargar_preguntas()
print(f"Preguntas cargadas: {len(filas)}")

In [ ]:
# TODO resuelto: hit@3 y hit@5 para chunk_size 400, 900 y 1500.
# Reconstruye el indice por cada chunk_size y evalua.
tabla = []
for cs in (400, 900, 1500):
    docs_cs = []
    for nombre, texto in textos.items():
        docs_cs.extend(pdf_to_documents(nombre, texto, chunk_size=cs,
                                        chunk_overlap=min(120, cs // 8)))
    vs_cs, _ = build_store(docs_cs, collection=f"eval_cs_{cs}")
    r3 = vs_cs.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    r5 = vs_cs.as_retriever(search_type="similarity", search_kwargs={"k": 5})
    h3, h5 = hit_at_k(r3, filas), hit_at_k(r5, filas)
    tabla.append((cs, len(docs_cs), h3, h5))
    print(f"chunk_size={cs:>4}  chunks={len(docs_cs):>4}  hit@3={h3:.2f}  hit@5={h5:.2f}")

print("\n| chunk_size | chunks | hit@3 | hit@5 |")
print("|---|---|---|---|")
for cs, nc, h3, h5 in tabla:
    print(f"| {cs} | {nc} | {h3:.2f} | {h5:.2f} |")
print("\nCopia esta tabla en RESPUESTAS.md (C4).")

## C6 · Persistencia del indice en disco

Cambio el cliente en memoria por uno con ruta. Si la coleccion no existe, indexo
y mido el tiempo; si existe, la reutilizo y mido el tiempo (debe ser mucho
menor). Tras reiniciar el runtime, esta celda debe responder sin reindexar.

In [ ]:
import time
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(path="./qdrant_data")
COL = "academic_papers_nvidia"
existe = qdrant_client.collection_exists(COL)
print("¿Ya existe la coleccion?", existe)

t0 = time.time()
if not existe:
    # requiere all_docs del notebook 03
    vs_disk, _ = build_store(all_docs, collection=COL, client=qdrant_client)
    print(f"Indexado desde cero en {time.time()-t0:.2f} s")
else:
    from langchain_qdrant import QdrantVectorStore
    vs_disk = QdrantVectorStore(client=qdrant_client, collection_name=COL, embedding=embeddings)
    print(f"Reutilizada sin reindexar en {time.time()-t0:.2f} s")

chain_disk = build_chain(vs_disk)
print(chain_disk.invoke("What is scaled dot-product attention?").strip())

# Comprobacion: reinicia el runtime, ejecuta solo Parte 0 (config) del nb 03 y
# esta celda; debe imprimir "Reutilizada sin reindexar" y responder igual.